# 02 — RAG Pipeline
**DS593 Final Project**

Q&A over Terms of Service and Privacy Policy documents.
Given a plain English question about any app, the system retrieves the relevant
clause and generates a grounded answer with a risk rating and citation.


## Step 0: Install Dependencies

In [ ]:
!pip install -q langchain langchain-community langchain-openai langchain-text-splitters chromadb openai
print("✓ Dependencies installed")


## Step 1: Configure API Key

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.getenv("OPENAI_API_KEY") and os.getenv("OPENAI_API_KEY").startswith("sk-"):
    print("✓ OpenAI API key configured")
else:
    raise ValueError("Missing OpenAI API key — set it in your venv/bin/activate")


## Step 2: Imports

In [ ]:
from typing import List
import chromadb
from chromadb.utils import embedding_functions
from langchain_openai import ChatOpenAI

print("✓ All imports successful")


## Step 3: Load ChromaDB Collection

Connects to the existing ChromaDB built in 01_chunk_index.ipynb.
No re-indexing needed.


In [ ]:
CHROMA_DIR  = "chroma_db"
EMBED_MODEL = "text-embedding-3-small"

openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.environ["OPENAI_API_KEY"],
    model_name=EMBED_MODEL,
)

client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = client.get_collection(
    name="tos_documents",
    embedding_function=openai_ef,
)

print(f"✓ Collection loaded: '{collection.name}'")
print(f"  Total chunks: {collection.count()}")


## Step 4: Initialize LLM

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.1,
    max_tokens=500,
)

print("✓ LLM initialized")


## Step 5: Retrieval Function

Retrieves the top-k most relevant chunks for a query, filtered by company.
Searches both ToS and Privacy Policy unless doc_type is specified.


In [ ]:
def retrieve_chunks(query: str, company: str, doc_type: str = None, k: int = 3) -> List[dict]:
    """Retrieve top-k relevant chunks from ChromaDB filtered by company."""
    if doc_type:
        where = {
            "$and": [
                {"company": {"$eq": company}},
                {"doc_type": {"$eq": doc_type}}
            ]
        }
    else:
        where = {"company": {"$eq": company}}

    results = collection.query(
        query_texts=[query],
        n_results=k,
        where=where,
    )

    chunks = []
    for doc, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        chunks.append({
            "text":     doc,
            "source":   meta["source"],
            "chunk":    meta["chunk_index"],
            "doc_type": meta["doc_type"],
            "distance": distance,
        })

    return chunks


# Test it
test_chunks = retrieve_chunks("Can Spotify sell my data?", company="spotify", k=2)
print(f"Retrieved {len(test_chunks)} chunks")
for c in test_chunks:
    print(f"  [{c['doc_type']}] chunk {c['chunk']} | distance: {c['distance']:.4f}")
    print(f"  {c['text'][:150]}...")
    print()


## Step 6: RAG Query Function

Complete pipeline: retrieve relevant chunks → build prompt → generate answer.
Returns answer, risk rating (RED/YELLOW/GREEN), and exact citation.


In [ ]:
def rag_query(question: str, company: str, doc_type: str = None, k: int = 3) -> dict:
    """Complete RAG pipeline — retrieve then generate."""
    try:
        # Step 1: Retrieve relevant chunks
        retrieved_chunks = retrieve_chunks(question, company=company, doc_type=doc_type, k=k)
        context = "\n\n".join([c["text"] for c in retrieved_chunks])
        sources = [f"{c['source']} (chunk {c['chunk']})" for c in retrieved_chunks]

        # Step 2: Build prompt
        prompt = f"""You are a legal analyst helping users understand Terms of Service and Privacy Policy documents.

Answer the user's question based ONLY on the provided document excerpts.
Be direct and specific. If the answer is not in the excerpts, say so.

After your answer, provide:
- Risk rating: RED (user-unfriendly), YELLOW (mixed), or GREEN (user-friendly)
- The exact clause or sentence that supports your answer

Document excerpts:
{context}

Question: {question}

Answer:"""

        # Step 3: Generate answer
        answer = llm.invoke(prompt).content

        return {
            "question": question,
            "company":  company,
            "chunks":   retrieved_chunks,
            "sources":  sources,
            "answer":   answer,
        }

    except Exception as e:
        print(f"Error: {e}")
        return {"error": str(e)}


# Test it
result = rag_query("Can Spotify sell my personal data to third parties?", company="spotify")
print(f"Question: {result['question']}")
print(f"Sources:  {result['sources']}")
print(f"\nAnswer:\n{result['answer']}")


## Step 7: Pretty Print Helper

In [ ]:
def ask(question: str, company: str, doc_type: str = None):
    """Wrapper for clean output."""
    print("=" * 70)
    print(f"Company:  {company.upper()}")
    print(f"Question: {question}")
    print("=" * 70)

    result = rag_query(question, company=company, doc_type=doc_type)

    if "error" in result:
        print(f"Error: {result['error']}")
        return

    print(f"\n{result['answer']}")
    print(f"\nSources: {', '.join(result['sources'])}")
    print()

print("✓ Helper ready")


## Step 8: Test Queries

In [ ]:
# Test 1: Data selling
ask("Can Spotify sell my personal data to third parties?", company="spotify")


In [ ]:
# Test 2: Account termination
ask("Can Discord delete my account without warning?", company="discord", doc_type="tos")


In [ ]:
# Test 3: Content ownership
ask("Does TikTok own the videos I upload?", company="tiktok")


In [ ]:
# Test 4: Arbitration
ask("Do I have to use arbitration instead of going to court?", company="netflix", doc_type="tos")


In [ ]:
# Test 5: Data deletion
ask("Can I request that LinkedIn delete all my data?", company="linkedin", doc_type="privacy")


## Step 9: Baseline Comparison

Compare RAG answer vs vanilla GPT-4o-mini with no retrieval.
This is the core evaluation story — does RAG actually improve answers?


In [ ]:
def baseline_query(question: str, company: str) -> str:
    """Vanilla LLM with no retrieval — for comparison."""
    prompt = f"""You are a legal analyst. Answer this question about {company}'s Terms of Service or Privacy Policy.
Be direct and specific. If you are not sure, say so.

Question: {question}

Answer:"""
    return llm.invoke(prompt).content


# Compare RAG vs baseline on same question
question = "Can Spotify sell my personal data to third parties?"
company  = "spotify"

print("=" * 70)
print("RAG ANSWER (grounded in actual document)")
print("=" * 70)
result = rag_query(question, company=company)
print(result["answer"])

print("\n" + "=" * 70)
print("BASELINE ANSWER (no retrieval)")
print("=" * 70)
baseline = baseline_query(question, company=company)
print(baseline)
